# KAIL — Notebook 02: H-Neuron Probe

> **Kora AI Lab** | H-AZR Research Pipeline

## What this notebook does

**Stage 2** — Identify the H-Neurons (hallucination-predictive neurons).

Key finding from literature: **fewer than 0.1%** of MLP neurons can predict whether the model is about to hallucinate. We identify these neurons by comparing MLP activations on hallucinated vs correct responses.

**Output:** `h_neurons.json` — used as penalty signal in Notebook 04 (H-AZR training).

**Runtime:** ~35 min on free Colab T4

In [ ]:
!pip install -q transformers>=4.40.0 bitsandbytes>=0.43.0 accelerate>=0.28.0 datasets tqdm

In [ ]:
import torch
import numpy as np
import json
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tqdm.notebook import tqdm

print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title Load model
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    output_hidden_states=True,
)
model.eval()
print(f"Loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# @title Collect MLP activations
def get_last_token_mlp_activations(prompt: str) -> dict:
    """Get MLP output activations at the last input token for each layer."""
    activations = {}

    def make_hook(layer_idx):
        def hook(module, inp, out):
            activations[layer_idx] = out.detach().cpu().float()
        return hook

    hooks = []
    for i, layer in enumerate(model.model.layers):
        hooks.append(layer.mlp.register_forward_hook(make_hook(i)))

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        model(**inputs)

    for h in hooks:
        h.remove()

    # Return last-token activations per layer: shape (hidden_dim,)
    return {k: v[0, -1, :].numpy() for k, v in activations.items()}


def get_model_answer(prompt: str, max_tokens: int = 80) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# @title Collect hallucinated vs correct activation pairs
dataset = load_dataset("trivia_qa", "rc.nocontext", split="validation")
samples = list(dataset.select(range(200)))

hall_acts = {}   # layer -> list of activation vectors (hallucinated)
corr_acts = {}   # layer -> list of activation vectors (correct)

n_hall = 0
n_corr = 0

for sample in tqdm(samples, desc="Probing"):
    question = sample["question"]
    answers = sample["answer"]["aliases"]  # all valid answers

    pred = get_model_answer(question)
    is_hall = not any(a.lower() in pred.lower() for a in answers)

    acts = get_last_token_mlp_activations(question)

    target = hall_acts if is_hall else corr_acts
    for layer_idx, act in acts.items():
        if layer_idx not in target:
            target[layer_idx] = []
        target[layer_idx].append(act)

    if is_hall: n_hall += 1
    else: n_corr += 1

print(f"\nHallucinated: {n_hall} | Correct: {n_corr}")

In [ ]:
# @title Identify H-Neurons via z-score
THRESHOLD = 2.0

h_neurons = {}
total_neurons = 0
total_h = 0
layer_h_counts = []

for layer_idx in hall_acts:
    if layer_idx not in corr_acts or not hall_acts[layer_idx] or not corr_acts[layer_idx]:
        continue

    hall_mean = np.mean(hall_acts[layer_idx], axis=0)
    corr_mean = np.mean(corr_acts[layer_idx], axis=0)

    diff = hall_mean - corr_mean
    std = np.std(diff)
    if std < 1e-8:
        continue

    z = diff / std
    h_idx = np.where(np.abs(z) > THRESHOLD)[0].tolist()

    total_neurons += len(diff)
    total_h += len(h_idx)
    layer_h_counts.append(len(h_idx))

    if h_idx:
        h_neurons[str(layer_idx)] = {
            "indices": h_idx,
            "z_scores": np.abs(z[h_idx]).tolist(),
        }

pct = 100 * total_h / total_neurons
print(f"H-Neurons identified: {total_h:,} / {total_neurons:,} ({pct:.4f}%)")
print(f"Affected layers: {len(h_neurons)} / {len(hall_acts)}")
print(f"Mean H-Neurons per layer: {np.mean(layer_h_counts):.1f}")

In [ ]:
# @title Visualize H-Neuron distribution across layers
layers = [int(k) for k in h_neurons.keys()]
counts = [len(h_neurons[str(l)]["indices"]) for l in layers]

plt.figure(figsize=(12, 4))
plt.bar(layers, counts, color="#1D9E75", alpha=0.8)
plt.xlabel("Layer index")
plt.ylabel("H-Neuron count")
plt.title(f"H-Neurons per layer ({MODEL_NAME})\nTotal: {total_h:,} ({pct:.4f}% of all neurons)")
plt.tight_layout()
plt.savefig("h_neuron_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

In [ ]:
# @title Save H-Neurons to JSON
output = {
    "model": MODEL_NAME,
    "threshold": THRESHOLD,
    "n_hall_samples": n_hall,
    "n_corr_samples": n_corr,
    "total_neurons": total_neurons,
    "total_h_neurons": total_h,
    "h_neuron_pct": pct,
    "layers": h_neurons,
}

with open("h_neurons.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved: h_neurons.json")
print(f"\nKey result: {pct:.4f}% of neurons are H-Neurons")
print("Upload h_neurons.json to the repo assets/ folder.")

## Summary

You now have `h_neurons.json` — the map of neurons that predict hallucination.

**This is a publishable result on its own:** the H-Neuron percentage + distribution in Qwen2.5-Coder-1.5B has likely not been reported before.

**Next steps:**
- Run `03_spin_warmup.ipynb` (Stage 1: SPIN)
- After SPIN: re-run this notebook to see if H-Neurons changed → **that's ablation data for the paper**

---
*KAIL — Kora AI Lab | github.com/kora-ai-lab*